# 🏗️ Code Atlas: Architecture & AI Walkthrough

This interactive notebook documents the complete architecture, algorithms, and AI models used in the **Code Atlas** system. 

## 📚 Modules Covered
1. **Core System**: Parallel processing & Worker engine
2. **AI Engine**: Multi-Model LLM Manager
3. **RAG Pipeline**: Hybrid Search, HyDE, & Reranking
4. **GraphRAG**: Code Knowledge Graph
5. **Optimization**: Indexing strategies

## 1. Setup & Imports
First, let's load the necessary modules from the `src` directory.

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project Root: {project_root}")

# Import Core Modules
from src.core.models import RepoConfig, Task
from src.core.worker import ParallelRepoWorker

# Import AI Modules
from src.ai.llm.manager import LLMManager
from src.ai.rag_enhanced import EnhancedRAGRetriever
from src.ai.graphrag import CodeGraph

## 2. Core Architecture: Parallel Worker

The core of the system is the `ParallelRepoWorker`. It manages multiple repositories and executes tasks in parallel using a process pool.

### Key Algorithms
- **Task Distribution**: Tasks are queued and assigned to worker processes.
- **State Management**: Tracks which tasks are `pending`, `in_progress`, or `completed`.
- **Optimization**: Uses `gc.collect()` and CPU throttling to prevent memory leaks.

In [ ]:
# Example: defining a Repository and a Task

repo = RepoConfig(
    name="code-atlas",
    local_path="./",
    gitlab_url="https://gitlab.com/example/code-atlas.git",
    description="The worker system itself"
)

task = Task(
    repo_name="code-atlas",
    task_id="DOCUMENT-001",
    description="Update documentation for AI module",
    files_to_modify=["docs/AI_ARCHITECTURE.md"]
)

print(f"Created Task: {task.task_id} for Repo: {repo.name}")

## 3. AI Architecture: Multi-Model Manager

The `LLMManager` handles interactions with AI providers. It implements a **Fallback Chain** algorithm.

### Algorithm: Fallback Chain
1. Try **Primary Provider** (e.g., OpenAI GPT-4).
2. If fail/timeout $\to$ Try **Secondary** (e.g., Claude Sonnet).
3. If fail $\to$ Try **Tertiary** (e.g., Gemini Pro).

### Models Supported
- `gpt-4o`, `gpt-3.5-turbo`
- `claude-3-5-sonnet`
- `gemini-1.5-pro`

In [ ]:
try:
    # Initialize Manager (will check env vars for keys)
    llm = LLMManager()
    
    print("Available Providers:", llm.get_available_providers())
    print("Fallback Chain:", llm.fallback_chain)
    
except Exception as e:
    print(f"LLM Manager initialization skipped (missing keys): {e}")

## 4. Enhanced RAG Pipeline

We use an **Enhanced RAG** (Retrieval-Augmented Generation) system that combines multiple search strategies.

### A. HyDE (Hypothetical Document Embeddings)
Instead of searching for the query directly, we generate a *hypothetical code snippet* that answers the query, then search for that code. This aligns the vector space (Code $\leftrightarrow$ Code).

### B. Hybrid Search Algorithm
Combines **Vector Search** (Semantic) with **BM25** (Keyword).

Logic:
```python
Final Score = (Vector Similarity * 0.7) + (BM25 Score * 0.3)
```

**Boosting**:
- Repo Name Match: +1.0
- File Path Match: +0.4
- Code Keyword Match: +0.2

### C. Cross-Encoder Reranking
Top-K results from Hybrid Search are re-scored using a Cross-Encoder model (`BAAI/bge-reranker-v2-m3`) which sees both the query and the document simultaneously for higher accuracy.

In [ ]:
# Example of initializing the RAG pipeline
# Note: Requires 'data/vector_db' to exist

try:
    rag = EnhancedRAGRetriever(
        vector_db_path="./data/vector_db",
        llm_manager=LLMManager(),  # Required for HyDE
        use_hyde=True,
        use_reranking=True,
        use_graphrag=True
    )
    
    print("✅ Enhanced RAG Pipeline Initialized")
    
except Exception as e:
    print(f"RAG initialization skipped (missing DB or keys): {e}")

## 5. GraphRAG: Code Knowledge Graph

Code is not just text; it's a graph. We map relationships between files and functions.

### Structure
- **Nodes**: `File`, `Function`, `Class`
- **Edges**: `Imports`, `Calls`, `Extends`

### Algorithm: Multi-Hop Retrieval
1. Find relevant function via Vector Search found `process_payment()`.
2. GraphRAG looks up edges: `process_payment()` calls `validate_card()`.
3. Retrieves `validate_card()` even if it didn't match the vector search.
4. Explains the full flow.

In [ ]:
# Demonstrating Graph Structure

graph = CodeGraph()

# Add nodes
graph.add_file("main.py", "core-repo", "python")
graph.add_function("main", "main.py", "core-repo")

graph.add_file("utils.py", "core-repo", "python")
graph.add_function("helper", "utils.py", "core-repo")

# Add relationship
graph.add_import("main.py", "utils.py", "core-repo")
graph.add_call("main.py", "main", "utils.py", "helper", "core-repo")

# Query neighbors
neighbors = graph.get_neighbors("core-repo:main.py")
print(f"Neighbors of main.py: {neighbors}")

## 6. Optimization Algorithms

To handle large codebases, we rely on **HNSW** (Hierarchical Navigable Small World) approximate nearest-neighbor search as implemented inside **Qdrant**. Collection and index parameters can be tuned via the Qdrant client when you need to trade memory for recall or build speed.

| Parameter | Example | Effect |
|-----------|---------|--------|
| `M` (connections) | tune per deployment | Higher M → more graph edges, more RAM |
| `ef_construct` | tune per deployment | Higher → better index quality, slower builds |

This allows the worker to process thousands of files on standard hardware when indexing and search settings are aligned with available RAM.